# Fairness Assessment & Mitigation

This notebook demonstrates how to measure biases across racial groups on the COMPAS dataset and apply ThresholdOptimizer post-processing to mitigate it.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

from configs.config import RANDOM_STATE, TEST_SIZE, TARGET_COLUMN, PROCESSED_PATH
from src.models.model_factory import ModelFactory
from src.fairness.threshold_optimizer import optimize_threshold
from src.fairness.metrics import get_fairness_summary

### Load Extended Dataset and Raw sensitive attributes

In [ ]:
processed_dir = Path("../") / PROCESSED_PATH
df = pd.read_csv(processed_dir / "compas_extended.csv")
df_raw = pd.read_csv(processed_dir / "compas_extended_raw.csv")

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

df_train_raw, df_test_raw = train_test_split(
    df_raw, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df_raw[TARGET_COLUMN]
)

### Fit Base Model and Measure Bias

In [ ]:
rf = ModelFactory.get_model("random_forest", random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

y_pred_base = rf.predict(X_test)
base_fairness = get_fairness_summary(y_test, y_pred_base, df_test_raw["race"])
print("Base Model Fairness metrics:")
for k, v in base_fairness.items():
    print(f"- {k}: {v:.4f}")

### Apply Post-Processing threshold mitigation

In [ ]:
mit_model = optimize_threshold(
    estimator=rf,
    X=X_train,
    y=y_train,
    sensitive_features=df_train_raw["race"],
    constraint="demographic_parity"
)

y_pred_mit = mit_model.predict(X_test, sensitive_features=df_test_raw["race"])
mit_fairness = get_fairness_summary(y_test, y_pred_mit, df_test_raw["race"])
print("Mitigated Model Fairness metrics:")
for k, v in mit_fairness.items():
    print(f"- {k}: {v:.4f}")